code

```

{
       "apikey": "tutaj-twój-klucz-api",
       "task": "people",
       "answer": [
         {
           "name": "Jan",
           "surname": "Kowalski",
           "gender": "M",
           "born": 1987,
           "city": "Warszawa",
           "tags": ["tag1", "tag2"]
         },
         {
           "name": "Anna",
           "surname": "Nowak",
           "gender": "F",
           "born": 1993,
           "city": "Grudziądz",
           "tags": ["tagA", "tagB", "tagC"]
         }
       ]
     }
```

In [1]:
import pathlib
from enum import Enum
import pydantic
import polars as pl
from loguru import logger

In [6]:
import sys
import os

sys.path.append("../..")

In [ ]:
from src.ai_devs_core import AIDevsClient, Config, get_config, JobClient, BatchJobConfig

TASK_NAME = "people"
DATA_SAVE_PATH = pathlib.Path("./data")


class Tag(Enum):
    IT = "IT"
    TRANSPORT = "transport"
    EDUKACJA = "edukacja"
    MEDYCYNA = "medycyna"
    PRACA_Z_LUDZMI = "praca z ludźmi"
    PRACA_Z_POJAZDAMI = "praca z pojazdami"
    PRACA_FIZYCZNA = "praca fizyczna"


class Classification(pydantic.BaseModel):
    classification: int
    tags: list[str]


def func_generating_dict(row):
    """Generate messages dict with id and job for Mistral batch API"""
    return [
        {
            "role": "system",
            "content": "You are a classification assistant. Analyze the job description and return ONLY a valid JSON response with the exact schema: {'classification': int, 'tags': list[str]}. Set classification to 1 if the person works in transportation, 0 if not, -1 if unsure. Use tags from this list only: ['IT', 'transport', 'edukacja', 'medycyna', 'praca z ludźmi', 'praca z pojazdami', 'praca fizyczna']. Never add explanations or text outside the JSON.",
        },
        {
            "role": "user",
            "content": f"Analyze this job description and return JSON only:\n\nID: {row['id']}\nJob: {row['job']}",
        },
    ]

In [ ]:

# Initialize configuration and clients
config: Config = get_config()
logger.info("Configuration loaded.")

ai_devs_core = AIDevsClient(
        api_url=config.AI_DEVS_API_URL, api_key=config.AI_DEVS_API_KEY
    )
job_client = JobClient(config)

# Read and filter data
logger.info("Reading and filtering data...")
df = ai_devs_core.get_dataset(dataset=TASK_NAME, save_path=DATA_SAVE_PATH)
logger.info(f"Initial data count: {len(df)}")
logger.info(df)


2026-03-26 13:27:34.405 | INFO     | __main__:<module>:3 - Configuration loaded.
2026-03-26 13:27:34.494 | WARNING  | src.ai_devs_core.job_client:_init_langfuse:157 - trace_name parameter not passed to JobClient, Observability disabled
2026-03-26 13:27:34.494 | INFO     | __main__:<module>:9 - Reading and filtering data...
2026-03-26 13:27:35.267 | INFO     | src.ai_devs_core.ai_devs_client:get_dataset:78 - Saving to data/people.csv
2026-03-26 13:27:35.345 | INFO     | __main__:<module>:11 - Initial data count: 24417
2026-03-26 13:27:35.346 | INFO     | __main__:<module>:12 - shape: (24_417, 7)
┌───────────┬─────────────┬────────┬────────────┬─────────────────┬──────────────┬─────────────────┐
│ name      ┆ surname     ┆ gender ┆ birthDate  ┆ birthPlace      ┆ birthCountry ┆ job             │
│ ---       ┆ ---         ┆ ---    ┆ ---        ┆ ---             ┆ ---          ┆ ---             │
│ str       ┆ str         ┆ str    ┆ str        ┆ str             ┆ str          ┆ str         

In [ ]:

# Calculate age and filter
df = df.select(
    pl.all(),
    pl.col("birthDate").str.to_datetime("%Y-%m-%d").dt.year().alias("born"),
    (
        pl.datetime(2024, 1, 1).dt.year()
        - pl.col("birthDate").str.to_datetime("%Y-%m-%d").dt.year()
    ).alias("age"),
)

df = df.with_row_index("id").filter(
    pl.col("gender") == "M",
    pl.col("age") >= 20,
    pl.col("age") <= 40,
    pl.col("birthPlace") == "Grudziądz",
)
logger.info(f"Filtered data count: {len(df)}")

# Prepare data for batch processing
df_with_messages = df.select(["id", "job"])
logger.info(f"Prepared {len(df_with_messages)} records for batch processing")



2026-03-26 13:27:43.102 | INFO     | __main__:<module>:17 - Filtered data count: 33
2026-03-26 13:27:43.104 | INFO     | __main__:<module>:21 - Prepared 33 records for batch processing


In [11]:

# Run batch job with enhanced configuration
logger.info("Starting batch job...")
result_df = job_client.batch_job(
    df=df_with_messages,
    schema=Classification,
    task=TASK_NAME,
    message_generator=func_generating_dict,
    config=BatchJobConfig(
        model="mistral-small-latest",
        poll_interval=5,
        timeout=120,
        max_workers=1,  # Set to >1 for parallel processing
        max_retries=5,
        chunk_size=1000,
        rate_limit=0,  # 0 = unlimited
    ),
)

# Log results
logger.info("Batch job completed!")
logger.info(f"Result DataFrame shape: {result_df.shape}")
logger.info(f"Result columns: {result_df.columns}")
logger.info("Sample results:")
logger.info(
    result_df.select(["id", "job", "classification", "tags", "_success"]).limit(10)
)

# Summary statistics
success_count = result_df.filter(pl.col("_success")).height
logger.info(f"Successfully processed: {success_count}/{len(result_df)}")

# Log metrics
metrics = job_client.get_metrics()
logger.info(f"Processing metrics: {metrics}")



2026-03-26 13:27:47.605 | INFO     | __main__:<module>:2 - Starting batch job...
2026-03-26 13:27:47.607 | INFO     | src.ai_devs_core.job_client:batch_job:282 - Starting batch job for task: people (correlation_id: c596bf2d-2d10-47d8-a63a-f8f6b434cb79)
2026-03-26 13:27:47.607 | INFO     | src.ai_devs_core.job_client:batch_job:285 - Input DataFrame shape: (33, 2)
2026-03-26 13:27:47.609 | INFO     | src.ai_devs_core.job_client:batch_job:299 - Generated 33 message sets
2026-03-26 13:27:47.610 | INFO     | src.ai_devs_core.job_client:_create_batch_job:767 - Launching a batch job, model=mistral-small-latest
2026-03-26 13:27:47.612 | INFO     | src.ai_devs_core.job_client:_create_batch_job:781 - Saved batch requests to data/batch_jobs/batch_requests_20260326_132747.json
2026-03-26 13:27:49.536 | INFO     | src.ai_devs_core.job_client:_process_with_retry:410 - Batch job created with ID: 472bab8a-fab9-445e-989a-461319aac5a4
2026-03-26 13:27:49.537 | INFO     | src.ai_devs_core.job_client:_wai

In [15]:
'''
{
    "name": "Jan",
    "surname": "Kowalski",
    "gender": "M",
    "born": 1987,
    "city": "Warszawa",
    "tags": ["tag1", "tag2"]
},
'''

final_df = (
    df.join(result_df, on="id")
    .filter(pl.col("classification") == 1)
    .select(
        pl.col("name"),
        pl.col("surname"),
        pl.col("gender"),
        pl.col("born"),
        pl.col("birthPlace").alias("city"),
        pl.col("tags"),
    )
)
final_df.to_dicts()

[{'name': 'Cezary',
  'surname': 'Żurek',
  'gender': 'M',
  'born': 1987,
  'city': 'Grudziądz',
  'tags': ['transport', 'praca z pojazdami']},
 {'name': 'Jacek',
  'surname': 'Nowak',
  'gender': 'M',
  'born': 1991,
  'city': 'Grudziądz',
  'tags': ['transport', 'praca z ludźmi']},
 {'name': 'Wojciech',
  'surname': 'Bielik',
  'gender': 'M',
  'born': 1986,
  'city': 'Grudziądz',
  'tags': ['transport', 'praca z pojazdami']},
 {'name': 'Wacław',
  'surname': 'Jasiński',
  'gender': 'M',
  'born': 1986,
  'city': 'Grudziądz',
  'tags': ['transport', 'praca z ludźmi']}]

In [ ]:
res = ai_devs_core.verify(task=TASK_NAME, data=final_df.to_dicts())
logger.info(f"Response from AI_devs API: {res}")